# Inter-Rater Agreement Study — Judge B Scoring
## VLM Medical VQA Benchmark

**Purpose:** Run a second, larger judge (Qwen3-30B-A3B via HuggingFace Inference API)
on the stratified 499-record sample produced by `scripts/build_inter_rater_sample.py`.
This validates whether the primary Llama-3.1-8B judge used throughout the benchmark
produces scores that agree with a significantly stronger model.

**Inputs:**
- `inter_rater_sample_500.jsonl` — uploaded as a Kaggle dataset (see setup notes)

**Outputs:**
- `inter_rater_results.jsonl` — each record has both `judge_score` (8B, already present)
  and `judge_b_score` (30B, computed here)

**No GPU required** — all inference is done via the HuggingFace Inference API.

---

### ⚠️ Setup before running
1. Upload `outputs/inter_rater_sample_500.jsonl` from your local machine as a
   Kaggle dataset named `inter-rater-sample`.
2. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`.
3. Run all cells. Expected runtime: ~25–40 minutes for 499 API calls.



## Cell 1 — Install dependencies


In [1]:
import subprocess
subprocess.run(['pip', 'install', 'huggingface_hub', 'tqdm', '-q'])
print('Done.')

Done.


## Cell 2 — Imports and HuggingFace login


In [2]:
import os, json, re, time
from tqdm.auto import tqdm
from huggingface_hub import InferenceClient, login

# Read HF token from Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('Logged in to HuggingFace.')



Logged in to HuggingFace.


## Cell 3 — Configure paths


In [3]:
# Input — uploaded Kaggle dataset
SAMPLE_PATH = '/kaggle/input/datasets/shriyanshraj/inter-rater-sample/inter_rater_sample_500.jsonl'

# Output — saved to working directory, then download manually
OUT_PATH = '/kaggle/working/inter_rater_results.jsonl'

# Checkpoint: resume if kernel was interrupted
print(f'Input:  {SAMPLE_PATH}')
print(f'Output: {OUT_PATH}')



Input:  /kaggle/input/datasets/shriyanshraj/inter-rater-sample/inter_rater_sample_500.jsonl
Output: /kaggle/working/inter_rater_results.jsonl


## Cell 4 — Load the stratified sample


In [4]:
records = []
for line in open(SAMPLE_PATH, encoding='utf-8'):
    records.append(json.loads(line))

print(f'Loaded {len(records)} records.')

# Quick breakdown
from collections import Counter
stratum_counts = Counter(r.get('sample_stratum', 'unknown') for r in records)
model_counts   = Counter(r.get('model', 'unknown').split('/')[-1] for r in records)

print('\nStrata:')
for k, v in sorted(stratum_counts.items()):
    print(f'  {k}: {v}')

print('\nModels:')
for k, v in sorted(model_counts.items()):
    print(f'  {k}: {v}')



Loaded 499 records.

Strata:
  medical_open: 150
  score_1_easy: 100
  score_3_ambiguous: 150
  score_5_easy: 99

Models:
  HuatuoGPT-Vision-7B-Qwen2.5VL: 96
  gemma-3-4b-it: 96
  llava-med-v1.5-mistral-7b: 113
  llava-v1.6-mistral-7b-hf: 95
  medgemma-4b-it: 99


## Cell 5 — Initialize Judge B (Qwen3-30B-A3B)

Using `Qwen/Qwen3-30B-A3B` — a 30B MoE model (~3B active parameters at inference)
available free on the HuggingFace Inference API.

**Critical:** We use the **identical prompt** as the Llama-3.1-8B judge. Only the model
changes. This is required for a valid inter-rater comparison.



In [6]:
# Exact same prompt used in 04_llm_judge.ipynb — do NOT modify
MEDICAL_JUDGE_PROMPT = """
You are an expert medical evaluator assessing the quality of answers to medical visual question answering (VQA) tasks.

You will be given:
- A medical question about a radiology or pathology image
- A reference answer (ground truth)
- A predicted answer from a vision-language model

Your task is to rate how correct the predicted answer is compared to the reference answer.
Focus on medical correctness and semantic equivalence, not exact wording.

Use this scale:
1: Completely wrong — the predicted answer is medically incorrect or entirely irrelevant
2: Mostly wrong — contains a relevant medical concept but misses the key point
3: Partially correct — captures the general idea but with a meaningful medical error or omission
4: Mostly correct — semantically equivalent to the reference with minor phrasing differences (e.g. 'Lungs' vs 'Lung')
5: Fully correct — matches the reference answer in medical meaning, possibly with different but equivalent phrasing

Provide your feedback as follows:

Feedback:::
Evaluation: (your medical reasoning for the rating, 1-2 sentences)
Total rating: (your rating, as a single integer between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question, reference answer, and predicted answer.

Question: {question}
Reference answer: {reference}
Predicted answer: {prediction}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """
print(f'Prompt length: {len(MEDICAL_JUDGE_PROMPT)} characters')



Prompt length: 1508 characters


## Cell 6 — Score extraction and smoke test


In [10]:
!pip install -q groq

import re
import time
from groq import Groq
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
# Ensure you added GROQ_API_KEY to your Kaggle Secrets!
groq_api_key = secrets.get_secret('GROQ_API_KEY')

# Initialize the Groq client
client = Groq(api_key=groq_api_key)

# Use Llama-3.3-70B as our massively scaled Judge B
JUDGE_B_MODEL = 'llama-3.3-70b-versatile'

def extract_judge_score(answer: str):
    try:
        if 'Total rating:' in answer:
            rating_text = answer.split('Total rating:')[1]
        else:
            rating_text = answer
        digits = re.findall(r'\d+(?:\.\d+)?', rating_text)
        if digits:
            score = float(digits[0])
            return max(1.0, min(5.0, score))
        return None
    except Exception as e:
        print(f'Extraction error: {e}')
        return None

def judge_b_single(question: str, reference: str, prediction: str, retries: int = 3) -> dict:
    prompt = MEDICAL_JUDGE_PROMPT.format(
        question=question,
        reference=reference,
        prediction=prediction,
    )
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=JUDGE_B_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=250,
                temperature=0.01
            )
            raw_text = response.choices[0].message.content
            score = extract_judge_score(raw_text)
            return {'judge_b_response': raw_text, 'judge_b_score': score}
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f'  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...')
                time.sleep(wait)
            else:
                return {'judge_b_response': str(e), 'judge_b_score': None}

# Smoke test
print('Smoke test on 3 known examples:')
test_cases = [
    ('What modality is used to take this image?', 'CT', 'Computed tomography (CT)', '~5'),
    ('What modality is used to take this image?', 'CT', 'MRI', '~1'),
    ('Which part of the body does this image belong to?', 'Chest', 'Chest/Thorax', '~4-5'),
]
for q, ref, pred, expected in test_cases:
    result = judge_b_single(q, ref, pred)
    print(f'  Q: {q[:60]}')
    print(f'  Ref: {ref}  |  Pred: {pred}  |  Expected: {expected}  |  Got: {result["judge_b_score"]}')
    print()

Smoke test on 3 known examples:
  Q: What modality is used to take this image?
  Ref: CT  |  Pred: Computed tomography (CT)  |  Expected: ~5  |  Got: 5.0

  Q: What modality is used to take this image?
  Ref: CT  |  Pred: MRI  |  Expected: ~1  |  Got: 1.0

  Q: Which part of the body does this image belong to?
  Ref: Chest  |  Pred: Chest/Thorax  |  Expected: ~4-5  |  Got: 5.0



## Cell 7 — Main evaluation loop

Iterates over all 499 records, calls Judge B for each, and appends results
to `inter_rater_results.jsonl` with checkpoint/resume support.

Expected runtime: ~25–40 minutes (approximately 3–5 seconds per API call).



In [11]:
# Load already-completed records (checkpoint resume)
completed_keys = set()
if os.path.exists(OUT_PATH):
    for line in open(OUT_PATH, encoding='utf-8'):
        r = json.loads(line)
        key = (r.get('model', ''), r.get('idx', 0), r.get('dataset', ''))
        completed_keys.add(key)
    print(f'Resuming: {len(completed_keys)} records already done.')
else:
    print('Starting fresh.')

# Run evaluation
failed = 0
with open(OUT_PATH, 'a', encoding='utf-8') as f_out:
    for record in tqdm(records, desc='Judge B scoring'):
        key = (record.get('model', ''), record.get('idx', 0), record.get('dataset', ''))
        if key in completed_keys:
            continue

        result = judge_b_single(
            question   = record.get('question', ''),
            reference  = record.get('ground_truth', ''),
            prediction = record.get('prediction', ''),
        )

        # Merge Judge B scores into the original record
        out_record = {**record, **result}
        f_out.write(json.dumps(out_record) + '\n')
        f_out.flush()

        if result['judge_b_score'] is None:
            failed += 1

        # Polite delay to stay within free-tier rate limits
        time.sleep(0.5)

print(f'\nDone. Total: {len(records)}  |  Failed (null score): {failed}')
print(f'Output saved to: {OUT_PATH}')

Resuming: 278 records already done.


Judge B scoring:   0%|          | 0/499 [00:00<?, ?it/s]


Done. Total: 499  |  Failed (null score): 0
Output saved to: /kaggle/working/inter_rater_results.jsonl


## Cell 8 — Quick validation of results


In [18]:
import numpy as np

results = [json.loads(l) for l in open(OUT_PATH)]
valid   = [r for r in results if r.get('judge_score') is not None and r.get('judge_b_score') is not None]

print(f'Total records in output:        {len(results)}')
print(f'Records with both scores:       {len(valid)}')
print(f'Records with null judge_b:      {len(results) - len(valid)}')

if valid:
    a = np.array([r['judge_score']   for r in valid])
    b = np.array([r['judge_b_score'] for r in valid])
    print(f'\n8B  judge — mean: {a.mean():.3f}, std: {a.std():.3f}')
    print(f'30B judge — mean: {b.mean():.3f}, std: {b.std():.3f}')
    print(f'Mean absolute difference: {np.mean(np.abs(a - b)):.3f}')
    print(f'Exact agreement: {np.mean(a.astype(int) == b.astype(int))*100:.1f}%')

    print('\n✅ Download inter_rater_results.jsonl from the Output tab.')
    print('   Place it in: outputs/inter_rater_results.jsonl')
    print('   Then run:    python3 scripts/inter_rater_analysis.py')



Total records in output:        499
Records with both scores:       448
Records with null judge_b:      51

8B  judge — mean: 3.163, std: 1.646
30B judge — mean: 3.076, std: 1.780
Mean absolute difference: 0.462
Exact agreement: 72.1%

✅ Download inter_rater_results.jsonl from the Output tab.
   Place it in: outputs/inter_rater_results.jsonl
   Then run:    python3 scripts/inter_rater_analysis.py
